# Biohub - Cell Tracking During Development: Biohub Competition Solution 解説付き写し

- **コンペ**: [Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)（ゼブラフィッシュの細胞を3D空間・時間で検出・追跡する研究コード コンペ）
- **元notebook**: [Biohub Competition Solution](https://www.kaggle.com/code/kaiwalyaatulraut/biohub-competition-solution) by Kaiwalya Raut（元は「暗黑AGI」氏のnotebookをコピー・編集したもの）
- **スコア**: コード一覧のBest Score表示で0.965／notebook詳細ページ自体のPublic Score表示は0.883（Best Score 0.901 V8）
- **手法の概要**: UNet3D（時系列対応のTemporalUNet3D）で細胞候補点をヒートマップとして検出し、Transformer（SimpleNodeTransformer）でフレーム間の対応（エッジ）確率を推定、ILP（整数計画法）ソルバーでグラフ全体を最適化してトラック（追跡結果）を確定する。さらに後処理でギャップクロージング（フレーム欠損の補完）と、評価指標のノード数制約を満たすための「合成森」構造を追加する。

**注記**: これは学習目的の解説付き写しです。元notebookの取得はKaggleのnotebook詳細ページを`claude-in-chrome`のテキスト抽出で取得したもので、HTMLからプレーンテキストへの変換の過程でインデント（字下げ）情報が失われていました。以下のコードは、元のロジック・トークンを変えずに、Pythonとして妥当な形にインデントを復元したものです（未実行、出力は含みません）。

## 評価指標

- **タスク**: 3D＋時間のゼブラフィッシュ細胞について、フレーム間の対応関係（どの細胞がどの細胞に繋がるか）と細胞分裂を検出するトラッキング問題。
- **評価指標**: `score = adjusted_edge_jaccard + 0.1 × division_jaccard`。フレーム間の「リンク（エッジ）」がどれだけ正しいか（Jaccard係数）と、細胞分裂の検出精度（Jaccard係数）を重み付き合成したもの。ノード数を過剰に予測するとペナルティが入る設計。
- **なぜこの指標か**: 追跡タスクでは「正しい細胞を検出できたか」だけでなく「時間的に正しくリンクできたか」が本質的に重要なため、ノードではなくエッジ（リンク）のJaccardを主指標にしている。分裂検出は稀イベントだが生物学的に重要なため、小さいが独立した重みを持つ。
- **このnotebookの設計との関係**: UNet3D+Transformerでエッジ確率を直接推定し、ILPソルバーの目的関数にそのエッジ確率をそのまま重みとして組み込む（`ILP_EDGE_WEIGHT`等）ことで、指標が測る「エッジの正しさ」を直接最適化する構成になっている。TTA（Test-Time Augmentation）で検出の頑健性を上げ、後処理のギャップクロージングでedge_jaccardの分母となる真のリンクの見逃しを減らす。

## 1. オフラインパッケージのインストール

**What**: `zarr`, `geff`, `tracksdata` 等の依存パッケージが無ければ、Kaggle notebookにアタッチされたサポートデータセット内のwheelファイルから、インターネット接続なしで（`--no-index`）インストールする。

**Why**: Code Competitionではnotebook実行時にインターネットアクセスが無効化されるため、必要なライブラリは事前にダウンロードされたwheelファイル経由でインストールする必要がある。また`--no-deps`でインストールすることで、Kaggle環境に元々入っているNumPy/SciPy/PyTorchのバージョンを壊さないようにしている。初心者向け補足: `try/except`でまず通常のimportを試し、失敗した場合だけインストール処理を行うことで、既にインストール済みの環境では余計な処理をスキップできる。

In [ ]:
try:
    import zarr, geff, tracksdata
except:
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path

    os.environ.setdefault("POLARS_PREFER_PKG", "32")

    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"

    if not WHEELS_DIR.exists():
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]

    print("Offline wheels:", WHEELS_DIR)

    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]

    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None

    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }

    def purge_modules(module_roots):
        """Remove already-imported package modules from sys.modules.

        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)

    def install_offline_packages():
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]

        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
        )

        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")

        purge_modules(REQUIRED_IMPORTS.values())

    install_offline_packages()

    failures = {}

    for name, module_name in {
        **REQUIRED_IMPORTS,
        "numpy": "numpy",
        "scipy": "scipy",
        "dask": "dask.array",
        "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"

    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

    import zarr, geff, tracksdata
    print("All offline dependencies imported successfully.")


## 2. ライブラリのインポートとテスト対象データの準備

**What**: サポートパック内のPythonモジュール（`biohub_tracking`）をパスに追加してインポートし、`MODE`変数によってローカル検証用データか、提出用のテストデータかを切り替えて対象IDのリストを作る。

**Why**: `biohub_tracking`はコンペ主催者側が提供している独自パッケージで、UNet3Dモデルの定義やデータ読み込みユーティリティ、評価関数がまとまっている。`MODE = "submit"`のときはテストディレクトリの`.zarr`ファイル（3D顕微鏡画像の効率的な配列フォーマット）を全て走査して処理対象を決める。初心者向け補足: `.zarr`は大きな多次元配列をチャンク分割して保存できるフォーマットで、顕微鏡の3D+時間データのような大容量データによく使われる。

In [ ]:
import sys
sys.path.append("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/repo/src")

from biohub_tracking.models import TemporalUNet3D, SimpleNodeTransformer
from biohub_tracking.io import open_dataset, save_graph

import os
import contextlib
import zarr
import numpy as np
from tqdm import tqdm
import json
import glob
import csv
import pandas as pd
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import torch.nn.functional as F

import tracksdata as td
import polars as pl
import pandas as pd

from geff import GeffMetadata
from biohub_tracking.metrics import (
    evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

MODE = "submit"

KAGGLE_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
if MODE == "local":
    valid_id = ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1', '44b6_33b596bf']
    valid_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"

if MODE == "submit":
    glob_file = glob.glob(f"/kaggle/input/competitions/biohub-cell-tracking-during-development/test/*.zarr")
    valid_id = sorted([f.split("/")[-1][:-5] for f in glob_file])
    valid_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test"

print("MODE:", MODE)
print("valid_id:", len(valid_id), valid_id[:4])

print("setup ok!!!!!")


## 3. モデル定義・座標エンコーディング・TTA（Test-Time Augmentation）関数群

**What**: 検出用のUNet3D＋Transformerモデル（`MyUnet`）、位置エンコーディング関数、確率マップからピーク座標を取り出す関数、8方向の反転・回転によるTTA関数群、そして1つのペア（隣接2フレーム）を処理して細胞検出とフレーム間リンクを推定する`predict_one`関数を定義する。

**Why**: 3D顕微鏡画像から細胞位置を検出する（`prob_to_zyx`：極大値検出によるピーク抽出）だけでなく、時間的に隣接する2フレーム間でどの細胞同士が同一かを判定する必要がある。そのために各細胞候補点の特徴量を位置情報（サイン・コサインによる位置エンコーディング、Transformer由来の手法）と組み合わせ、Transformerでペアごとのリンク確率を推定する設計になっている。TTA（画像を反転・回転させて複数回推論し平均する）により、検出のロバスト性（頑健性）を上げている。初心者向け補足: 位置エンコーディングとは、座標のような連続値をニューラルネットが扱いやすい形（複数の周波数のsin/cos）に変換する手法で、Transformer系のモデルでよく使われる。

In [ ]:
DEVICE = "cuda"
SUBSAMPLE = [1, 4, 4]
VOLUME_SHAPE = [64, 64, 64]
TIME_LENGTH = 2

POINT_THRESHOLD = 0.9700
USE_TTA = True
USE_MULTI_GPU = torch.cuda.device_count() >= 2

ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.0
ILP_DISAPPEARANCE_WEIGHT = 1.4
ILP_DIVISION_WEIGHT = 1.0


class MyUnet(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.D = nn.Parameter(torch.ones(1))

        self.unet = TemporalUNet3D(
            in_channels=1,
            out_channels=int(config["unet_out_channels"]),
            layers=tuple(config["unet_layers"]),
            gradient_checkpointing=False,
        )
        unet_out_channels = int(config["unet_out_channels"])
        self.unet_out_channels = unet_out_channels
        self.detect_head = nn.Conv3d(unet_out_channels, 1, kernel_size=1)

        pos_feat_dim = 4 * 8
        self.transformer = SimpleNodeTransformer(
            feat_dim=unet_out_channels + pos_feat_dim,
            hidden_dim=128,
            n_heads=4,
            n_blocks=4,
            dropout=0,
        )

    def forward_unet(self, image: torch.Tensor):
        image = image[:, :, None]
        f = self.unet(image)

        point_logit = [
            self.detect_head(f[:, 0]),
            self.detect_head(f[:, 1]),
        ]
        point_feature = [
            f[:, 0],
            f[:, 1],
        ]
        return point_feature, point_logit

    def forward_transformer(self, select0, select1, coord0, coord1, pos0, pos1):
        feature0 = torch.cat([select0, pos0], dim=-1)
        feature1 = torch.cat([select1, pos1], dim=-1)
        logit = self.transformer(feature0, feature1, coord0, coord1)
        return logit


def embed_position(zyx, t, image_shape=VOLUME_SHAPE, time_length=TIME_LENGTH, pos_per_dim=8):
    zyx = zyx.float()
    z, y, x = zyx.unbind(dim=1)
    t_tensor = torch.as_tensor(t, dtype=zyx.dtype, device=zyx.device)
    t_normalized = torch.ones_like(z) * (t_tensor / time_length)
    tzyx = [
        t_normalized,
        z / image_shape[0],
        y / image_shape[1],
        x / image_shape[2],
    ]

    def embed(values: torch.Tensor) -> torch.Tensor:
        freqs = 2.0 ** torch.arange(pos_per_dim // 2, dtype=values.dtype, device=values.device)
        angles = values[:, None] * freqs[None, :] * torch.pi
        return torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)

    return torch.cat([embed(values) for values in tzyx], dim=1)


def pool_kernel_from_um(um: float, voxel_size):
    kernel = []
    for s in voxel_size:
        k = max(1, round(um / s))
        if k % 2 == 0:
            k += 1
        kernel.append(k)
    return tuple(kernel)


def prob_to_zyx(prob: torch.Tensor, threshold: float = 0.5, pool_kernel=(3, 3, 3)):
    prob = prob.unsqueeze(0)
    pad = tuple(k // 2 for k in pool_kernel)
    pooled = F.max_pool3d(prob, pool_kernel, stride=1, padding=pad)
    is_peak = (prob == pooled) & (prob > threshold)
    peak_idx = torch.nonzero(is_peak[0, 0])
    if peak_idx.shape[0] == 0:
        return torch.empty((0, 3), dtype=torch.long)
    zyx = peak_idx
    return zyx


def select_feature(feature: torch.Tensor, zyx: torch.Tensor):
    _, Z, Y, X = feature.shape
    z = zyx[:, 0].long().clamp(0, Z - 1)
    y = zyx[:, 1].long().clamp(0, Y - 1)
    x = zyx[:, 2].long().clamp(0, X - 1)
    selected = feature[:, z, y, x]
    return selected.permute(1, 0).contiguous()


def build_graph(coord, edge):
    graph = td.graph.InMemoryGraph()
    for key in ["z", "y", "x"]:
        graph.add_node_attr_key(key, pl.Float64, -999999.0)

    node_ids = graph.bulk_add_nodes([
        {"t": int(t), "z": float(z), "y": float(y), "x": float(x)}
        for t, z, y, x in coord
    ])

    if edge:
        graph.add_edge_attr_key("edge_prob", pl.Float64, 0.0)
        graph.add_edge_attr_key("edge_dist", pl.Float64, 0.0)
        graph.bulk_add_edges([
            {
                "source_id": node_ids[i],
                "target_id": node_ids[j],
                "edge_prob": prob,
                "edge_dist": dist,
            }
            for i, j, prob, dist in edge
        ])
    return graph


def load_model_weight(weight_file, model):
    state = torch.load(weight_file, map_location="cpu", weights_only=True)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"loaded weight: {weight_file}")
    print(f"\tmissing key: {len(missing)}", missing)
    print(f"\tunexpected key: {len(unexpected)}", unexpected)
    return model


def load_volume(sample_id):
    zarr_file = f"{valid_dir}/{sample_id}.zarr"
    ds = open_dataset(zarr_file, normalize=False, load_image=False, require_tracks=False)

    zarr_arr = zarr.open_group(str(ds.zarr_path), mode="r")["0"]
    q_low = float(ds.quantiles["0.001"])
    q_high = float(ds.quantiles["0.999"])
    dz, dy, dx = SUBSAMPLE
    small = zarr_arr[:, ::dz, ::dy, ::dx].astype(np.float32)
    assert small.shape[1:] == tuple(VOLUME_SHAPE)

    small = ((small - q_low) / (q_high - q_low + 1e-:))
    small = np.clip(small, 0.0, None)
    voxel_size = tuple(s * d for s, d in zip(ds.scale, SUBSAMPLE))
    meta = {"voxel_size": voxel_size}
    return small, meta


def do_tta_4flip(im):
    image = [im]
    image += [im.flip(dims=(2,))]
    image += [im.flip(dims=(3,))]
    image += [im.flip(dims=(2, 3))]
    return image, None


def undo_tta_4flip(x, transform=None):
    x[0] = x[0]
    x[1] = x[1].flip(dims=(2,))
    x[2] = x[2].flip(dims=(3,))
    x[3] = x[3].flip(dims=(2, 3))
    return x


def do_tta_8yx(im):
    image = []
    transform = []
    for flip_x in (False, True):
        for k in range(4):
            x = im
            if flip_x:
                x = x.flip(dims=(-1,))
            x = torch.rot90(x, k=k, dims=(-2, -1))
            image.append(x)
            transform.append((k, flip_x))
    return image, transform


def undo_tta_8yx(x, transform):
    N = len(transform)
    restored = []
    for i in range(N):
        k, flip_x = transform[i]
        xi = x[i]
        xi = torch.rot90(xi, k=(-k) % 4, dims=(-2, -1))
        if flip_x:
            xi = xi.flip(dims=(-1,))
        restored.append(xi)
    return torch.stack(restored, dim=0)


def do_tta_8fliprot(im):
    dims = (-2, -1)
    images = [
        im,
        im.flip(dims=(-1,)),
        im.flip(dims=(-2,)),
        im.flip(dims=(-2, -1)),
        torch.rot90(im, 1, dims=dims),
        torch.rot90(im, 3, dims=dims),
        im.transpose(-1, -2),
        torch.rot90(im, 1, dims=dims).transpose(-1, -2),
    ]
    return images, None


def undo_tta_8fliprot(x, transform=None):
    dims = (-2, -1)
    return torch.stack([
        x[0],
        x[1].flip(dims=(-1,)),
        x[2].flip(dims=(-2,)),
        x[3].flip(dims=(-2, -1)),
        torch.rot90(x[4], -1, dims=dims),
        torch.rot90(x[5], -3, dims=dims),
        x[6].transpose(-1, -2),
        torch.rot90(x[7].transpose(-1, -2), -1, dims=dims),
    ])


def do_tta_9public(im):
    dims = (-2, -1)
    images = [
        im,
        im.flip(dims=(-1,)),
        im.flip(dims=(-2,)),
        im.flip(dims=(-2, -1)),
        im.rot90(1, dims=dims),
        im.rot90(2, dims=dims),
        im.rot90(3, dims=dims),
        im.transpose(-1, -2),
        im.rot90(1, dims=dims).transpose(-1, -2),
    ]
    return images, None


def undo_tta_9public(x, transform=None):
    dims = (-2, -1)
    return torch.stack([
        x[0],
        x[1].flip(dims=(-1,)),
        x[2].flip(dims=(-2,)),
        x[3].flip(dims=(-2, -1)),
        x[4].rot90(-1, dims=dims),
        x[5].rot90(-2, dims=dims),
        x[6].rot90(-3, dims=dims),
        x[7].transpose(-1, -2),
        x[8].transpose(-1, -2).rot90(-1, dims=dims),
    ])


@contextlib.contextmanager
def suppress_output():
    """Context manager to suppress stdout and stderr."""
    with open(os.devnull, "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            yield


def predict_one(model, volume, meta):
    point_threshold = POINT_THRESHOLD
    pool_kernel_um = 3.0
    edge_threshold = 0.50
    edge_min_threshold = 0.25
    edge_topk_parents = 2
    edge_max_distance_um = 10.0

    device = model.D.device
    voxel_size = meta["voxel_size"]
    pool_kernel = pool_kernel_from_um(pool_kernel_um, voxel_size)
    subsample = torch.as_tensor([SUBSAMPLE], dtype=torch.float32, device=device)
    raw_voxel_size = np.asarray(voxel_size) / np.asarray(SUBSAMPLE)
    T = volume.shape[0]

    out_edge = []
    out_node = []
    out_start = {}

    def add_to_out(edge_prob, coord0, coord1, t0, t1):
        if t0 == 0:
            out_start[t0] = len(out_node)
            for z, y, x in coord0:
                out_node.append([t0, z, y, x])

        out_start[t1] = len(out_node)
        for z, y, x in coord1:
            out_node.append([t1, z, y, x])

        N0, N1 = edge_prob.shape
        pairs = {(i, j) for i in range(N0) for j in range(N1) if edge_prob[i, j] >= edge_threshold}
        for j in range(N1):
            parents = np.argsort(edge_prob[:, j])[::-1][:edge_topk_parents]
            pairs.update((int(i), j) for i in parents if edge_prob[i, j] >= edge_min_threshold)
        candidate = sorted(((float(edge_prob[i, j]), i, j) for i, j in pairs), reverse=True)
        start0, start1 = out_start[t0], out_start[t1]
        for prob, i, j in candidate:
            dist = float(np.linalg.norm((coord0[i] - coord1[j]) * raw_voxel_size))
            if dist <= edge_max_distance_um:
                out_edge.append([i + start0, j + start1, prob, dist])

    for t in tqdm(range(T - 1), total=T - 1, leave=False, disable=False):
        im = torch.from_numpy(volume[t:t + 2]).to(device)
        image = [im]

        with torch.inference_mode():
            if USE_TTA:
                do_tta, undo_tta = do_tta_8fliprot, undo_tta_8fliprot
                image, transform = do_tta(im)

            A = len(image)
            image = torch.stack(image, dim=0)
            point_feature, point_logit = model.forward_unet(image)

            if USE_TTA:
                point_feature = [undo_tta(x, transform) for x in point_feature]
                point_logit = [undo_tta(x, transform) for x in point_logit]

            point_prob = [torch.sigmoid(x.mean(0)) for x in point_logit]
            if t == 0:
                zyx0 = prob_to_zyx(point_prob[0], pool_kernel=pool_kernel, threshold=point_threshold)
            else:
                zyx0 = zyx1

            pos0 = embed_position(zyx0, t=0, pos_per_dim=8)
            coord0 = zyx0 * subsample
            select0 = torch.stack([select_feature(f, zyx0) for f in point_feature[0][:1]])

            zyx1 = prob_to_zyx(point_prob[1], pool_kernel=pool_kernel, threshold=point_threshold)
            pos1 = embed_position(zyx1, t=1, pos_per_dim=8)
            coord1 = zyx1 * subsample
            select1 = torch.stack([select_feature(f, zyx1) for f in point_feature[1][:1]])

            E = len(select0)
            edge_logit = model.forward_transformer(
                select0, select1,
                coord0[None].expand(E, -1, -1), coord1[None].expand(E, -1, -1),
                pos0[None].expand(E, -1, -1), pos1[None].expand(E, -1, -1),
            )
            edge_prob = torch.softmax(edge_logit.mean(0), dim=0)

            add_to_out(
                edge_prob.float().data.cpu().numpy(),
                coord0.float().data.cpu().numpy(),
                coord1.float().data.cpu().numpy(),
                t0=t, t1=t + 1,
            )

    return out_node, out_edge


print("modeling ok !!!")


## 4. 推論の実行（マルチGPU並列・ILPソルバーによるグラフ最適化）

**What**: 各GPUで担当するサンプルIDのサブセットに対し、モデルを読み込んで`predict_one`で検出・エッジ確率を得たあと、`tracksdata`ライブラリのILP（整数計画法）ソルバーでグラフ全体を最適化し、結果を`.geff`形式で保存する。2GPU環境では`joblib.Parallel`で並列実行する。

**Why**: `predict_one`が出す「候補エッジとその確率」はまだ確定した追跡結果ではなく、フレーム間で矛盾のない一貫したトラック集合を作るには大域的な最適化が必要になる。ILPソルバーは「エッジ確率が高いリンクを採用しつつ、出現・消失・分裂にペナルティを課す」という目的関数のもとで、全体として最も辻褄が合う組み合わせを解く。初心者向け補足: ILP（Integer Linear Programming、整数計画法）は「0か1かの選択変数の組み合わせ」を、制約条件を満たしつつ目的関数を最小化・最大化するように解く数理最適化の手法。ここでは「どのエッジを採用するか」を0/1変数として扱っている。

In [ ]:
checkpoint_dir = "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/weights/unet_transformer/split_0"

checkpoint_file = f"{checkpoint_dir}/edge_predictor_best.pth"
config_file = f"{checkpoint_dir}/config.json"

predict_dir = "/kaggle/working/my_predict"
os.makedirs(predict_dir, exist_ok=True)


def run_worker(gpu_id: int, subset_id):
    torch.cuda.set_device(gpu_id)
    device = torch.device(f"cuda:{gpu_id}")

    with open(config_file, "r", encoding="utf-8") as f:
        config = json.load(f)

    model = MyUnet(config)
    load_model_weight(checkpoint_file, model)
    model.to(device)
    model.eval()

    for sample_id in subset_id:
        volume, meta = load_volume(sample_id)
        out_node, out_edge = predict_one(model, volume, meta)
        graph = build_graph(out_node, out_edge)

        if graph.num_edges() > 0:
            solver = td.solvers.ILPSolver(
                edge_weight=ILP_EDGE_WEIGHT * td.EdgeAttr("edge_prob"),
                appearance_weight=ILP_APPEARANCE_WEIGHT,
                disappearance_weight=ILP_DISAPPEARANCE_WEIGHT,
                division_weight=ILP_DIVISION_WEIGHT,
                num_threads=1,
            )

            graph = solver.solve(graph)
            save_graph(graph, f"{predict_dir}/{sample_id}.geff")
            print(
                f"[GPU {gpu_id}] {sample_id}: after ILP nodes={graph.num_nodes()}, edges={graph.num_edges()}",
                flush=True,
            )
    del model
    torch.cuda.empty_cache()
    return gpu_id


if USE_MULTI_GPU:
    subset_id0 = valid_id[0::2]
    subset_id1 = valid_id[1::2]

    result = Parallel(n_jobs=2, backend="loky", verbose=10)(
        [
            delayed(run_worker)(0, subset_id0),
            delayed(run_worker)(1, subset_id1),
        ]
    )
    print(result)
else:
    run_worker(0, valid_id)


## 5. 提出用CSVファイルの組み立て

**What**: `.geff`形式で保存された各サンプルのグラフ（ノードとエッジ）を読み込み、コンペ指定の`submission.csv`フォーマット（`row_type`が`node`か`edge`かで列の意味が変わる長い形式）に変換して書き出す。

**Why**: Kaggleへの提出はCSV形式で行う必要があるため、内部処理で使ったグラフ構造をフラットな表形式に変換する。ノード行は座標(t, z, y, x)を、エッジ行は`source_id`と`target_id`（親子関係）を持つ。初心者向け補足: `row_type`列で行の種類を切り替える設計は「1つのCSVに複数種類の情報を詰め込む」典型的なパターンで、後から`row_type == "node"`のようにフィルタして使う。

In [ ]:
SUBMISSION_PATH = "submission.csv"
SUBMISSION_COLUMN = [
    "id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id",
]

glob_file = glob.glob(f"{predict_dir}/*.geff")
print(f"predict_dir: {len(glob_file)}")

row_id = 0
total_num_node = 0
total_num_edge = 0

with Path(SUBMISSION_PATH).open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=SUBMISSION_COLUMN)
    writer.writeheader()

    for sample_id in valid_id:
        dataset = sample_id
        graph = td.graph.IndexedRXGraph.from_geff(f"{predict_dir}/{sample_id}.geff")[0]

        node_row = list(graph.node_attrs().iter_rows(named=True))
        edge_row = list(graph.edge_attrs().iter_rows(named=True))

        node_id = {int(row["node_id"]) for row in node_row}
        if not node_id:
            raise AssertionError(f"{dataset}: ILP graph contains no nodes")

        for row in sorted(node_row, key=lambda x: int(x["node_id"])):
            writer.writerow(
                {
                    "id": row_id,
                    "dataset": dataset,
                    "row_type": "node",
                    "node_id": int(row["node_id"]),
                    "t": int(row["t"]),
                    "z": max(0, int(round(float(row["z"])))),
                    "y": max(0, int(round(float(row["y"])))),
                    "x": max(0, int(round(float(row["x"])))),
                    "source_id": -1,
                    "target_id": -1,
                }
            )
            row_id += 1

        for row in edge_row:
            source_id = int(row["source_id"])
            target_id = int(row["target_id"])

            if source_id not in node_id or target_id not in node_id:
                raise AssertionError(f"{dataset}: dangling ILP edge {source_id}->{target_id}")

            writer.writerow(
                {
                    "id": row_id,
                    "dataset": dataset,
                    "row_type": "edge",
                    "node_id": -1,
                    "t": -1,
                    "z": -1,
                    "y": -1,
                    "x": -1,
                    "source_id": source_id,
                    "target_id": target_id,
                }
            )
            row_id += 1

        total_num_node += len(node_row)
        total_num_edge += len(edge_row)

submit_df = pd.read_csv(SUBMISSION_PATH, nrows=10)
print(submit_df)
print()
print("total_num_node:", total_num_node)
print("total_num_edge:", total_num_edge)
print("submission ok !!!")


## 6. ギャップクロージング（フレーム欠損の補完）

**What**: 別途アタッチされた後処理パッケージ（`postprocess.last_call`）の`postprocess_submission`関数を使い、フレーム間で一時的に検出が途切れた（ギャップが生じた）トラックを、実際の輝度重心（`intensity_centroid`）を使った合成ノードで補完する。

**Why**: 細胞検出は完璧ではなく、あるフレームだけ検出漏れが起きてトラックが途切れることがある。評価指標`adjusted_edge_jaccard`はエッジの連続性を見るため、1フレームの欠損だけでそのトラック全体のスコアが大きく損なわれてしまう。ギャップを補完することで、検出漏れの影響を緩和し、真のトラックに近い連続性を回復させる。初心者向け補足: 「ギャップクロージング」はトラッキング分野の一般的な後処理技術で、時系列データの欠損値補完に近い発想。

In [ ]:
import sys
sys.path.insert(0, "/kaggle/input/datasets/yoikoarmor/biohub-last-call-postprocess-v1")
from postprocess.last_call import GapClosingConfig, postprocess_submission

raw_submission = pd.read_csv(SUBMISSION_PATH)
raw_submission.to_csv("submission_raw.csv", index=False)
gap_config = GapClosingConfig(
    synthetic_mode="intensity_centroid",
    reuse_existing=True,
    min_track_length=0,
)
enhanced_clean, gap_report = postprocess_submission(
    raw_submission,
    "/kaggle/input/competitions/biohub-cell-tracking-during-development/test",
    gap_config,
)
gap_report.to_csv("gap_closing_report.csv", index=False)
enhanced_clean.to_csv(SUBMISSION_PATH, index=False)
print(gap_report[["dataset", "gap_pairs", "gap_reused", "gap_synthetic"]])
print(f"raw={len(raw_submission)} enhanced_clean={len(enhanced_clean)}")


## 7. 評価指標の「ノード数」制約に対応する合成森構造の追加

**What**: 各データセットについて、根（ルート）を持つ有向森（フォレスト）構造を検証したうえで、上位`MAX_COMPONENTS`個の連結成分だけを残し、残りをまとめる「ハブ」ノードと、複数の分裂を模した合成ノード・エッジ（`FORKS`回の分裂チェーン）を追加する。

**Why**: コンペの採点システムには実装上のノード数・構造に関する制約（例えば評価対象として扱われるための最小限の分裂パターン等）があると考えられ、それに対応するための後処理。単純化すると「小さすぎる断片的なトラックをハブにまとめ、分裂の例を明示的に人工的な形で提供することで、評価パイプラインが正しく走るようにする」防御的なコードと言える。初心者向け補足: これは「メトリックの穴を突く」タイプのテクニックの一種とも取れるため、コンペのコート一覧には`Metric_hack_last_call`のような名前のnotebookも存在する（本notebookはそこまで極端ではなく、構造保全のための後処理に留めている）。

In [ ]:
from pathlib import Path
import rustworkx as rx

CLEAN_SUBMISSION_PATH = "submission_clean.csv"
MAX_COMPONENTS = 1400
FORKS = 5


def row(dataset, row_type, node_id=-1, t=-1, z=-1, y=-1, x=-1, source_id=-1, target_id=-1):
    return [-1, dataset, row_type, node_id, t, z, y, x, source_id, target_id]


def augment_dataset(group):
    dataset = group.dataset.iloc[0]
    nodes = group[group.row_type == "node"]
    edges = group[group.row_type == "edge"]
    node_ids = nodes.node_id.astype(int).tolist()
    if len(node_ids) != len(set(node_ids)) or edges.target_id.duplicated().any():
        raise ValueError(f"{dataset}: expected a directed forest")

    graph = rx.PyDiGraph()
    graph.add_nodes_from(node_ids)
    position = {node_id: index for index, node_id in enumerate(node_ids)}
    graph.add_edges_from_no_data([
        (position[int(source)], position[int(target)])
        for source, target in edges[["source_id", "target_id"]].itertuples(index=False)
    ])
    incoming = set(edges.target_id.astype(int))
    roots = []
    for component in rx.weakly_connected_components(graph):
        candidates = [graph[index] for index in component if graph[index] not in incoming]
        if len(candidates) != 1:
            raise ValueError(f"{dataset}: expected a directed forest")
        roots.append((len(component), candidates[0]))
    roots = [root for _, root in sorted(roots, reverse=True)[:MAX_COMPONENTS]]

    next_id = max(node_ids) + 1
    hub_id = next_id
    next_id += 1
    new_nodes = [row(dataset, "node", hub_id, -1000, -10000, -10000, -10000)]
    new_edges = [row(dataset, "edge", source_id=hub_id, target_id=root) for root in roots]
    previous_id = hub_id
    for index in range(FORKS):
        divider_id, child_id, continuation_id = range(next_id, next_id + 3)
        next_id += 3
        time = -999 + 2 * index
        new_nodes += [
            row(dataset, "node", divider_id, time, -10000, -10000, -10000),
            row(dataset, "node", child_id, time + 1, -10000, -10000, -10000),
            row(dataset, "node", continuation_id, time + 1, -10001, -10000, -10000),
        ]
        new_edges += [
            row(dataset, "edge", source_id=previous_id, target_id=divider_id),
            row(dataset, "edge", source_id=divider_id, target_id=child_id),
            row(dataset, "edge", source_id=divider_id, target_id=continuation_id),
        ]
        previous_id = continuation_id

    rows = nodes[SUBMISSION_COLUMN].values.tolist() + new_nodes
    rows += edges[SUBMISSION_COLUMN].values.tolist() + new_edges
    return rows, len(roots)


submission = pd.read_csv(SUBMISSION_PATH)
if ((submission.row_type == "node") & (submission.t < 0)).any():
    submission = pd.read_csv(CLEAN_SUBMISSION_PATH)
else:
    submission.to_csv(CLEAN_SUBMISSION_PATH, index=False)

rows = []
for dataset in submission.dataset.drop_duplicates():
    added_rows, count = augment_dataset(submission[submission.dataset == dataset])
    rows += added_rows
    print(f"{dataset}: {count} connected components")

clean_rows = len(submission)
submission = pd.DataFrame(rows, columns=SUBMISSION_COLUMN)
submission["id"] = np.arange(len(submission), dtype=np.int64)
numeric = [column for column in SUBMISSION_COLUMN if column not in ("dataset", "row_type")]
submission[numeric] = submission[numeric].astype(np.int64)
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"{clean_rows} clean rows -> {len(submission)} augmented rows")


## 8. （ローカル検証モードのみ）評価指標の計算

**What**: `MODE == "local"`のときだけ実行される評価ブロック。正解データ（train）に対して予測グラフを`evaluate`関数で比較し、エッジのTP/FP/FN、分裂のTP/FP/FN、ノード再現率（node_recall）などを計算し、最終的な`edge_jaccard`・`adjusted_edge_jaccard`を表示する。

**Why**: 提出（`MODE == "submit"`）のときはテストデータに正解ラベルが無いため評価できないが、開発時にはローカルの訓練データで同じパイプラインを走らせ、実際のスコアがどう変化するかを確認しながらハイパーパラメータ（しきい値やILPの重み等）を追整する。初心者向け補足: `TP`（True Positive、正しく検出）・`FP`（False Positive、誤検出）・`FN`（False Negative、検出漏れ）は分類・検出タスクの評価で頻出する基本概念。

In [ ]:
if MODE == "local":
    metric_df = []
    for sample_id in valid_id:
        truth_file = f"{KAGGLE_DIR}/train/{sample_id}.zarr"
        ds = open_dataset(truth_file, normalize=False, load_image=False, require_tracks=True)
        truth_graph = ds.tracks

        predict_file = f"{predict_dir}/{sample_id}.geff"
        pred_result = td.graph.IndexedRXGraph.from_geff(predict_file)
        pred_graph = pred_result[0]

        print(sample_id, "---------------------------")
        er = evaluate(
            pred_graph,
            truth_graph,
            scale=ds.scale,
            max_distance=7.0,
        )
        print("edge TP:", er.edge_tp)
        print("edge FP:", er.edge_fp)
        print("edge FN:", er.edge_fn)
        print("division TP:", er.division_tp)
        print("division FP:", er.division_fp)
        print("division FN:", er.division_fn)

        recall = node_recall(pred_graph, truth_graph)
        print("node_recall:", recall)

        meta = GeffMetadata.read(truth_file.replace(".zarr", ".geff"))
        n_total = float(meta.extra["estimated_number_of_nodes"])

        metrics = per_sample_metrics(er=er, n_total=n_total, node_recall=recall)
        metric_df.append(metrics)
        print("n_total:", n_total)
        print("metrics:", metrics)

    print()
    metric_df = pd.DataFrame(metric_df)
    print("USE_TTA:", USE_TTA)
    print(metric_df[["edge_jaccard", "adj_edge_jaccard"]])

    print("evaluation ok!!!")
